In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import requests
import os

In [2]:
batch_size = 64
block_size = 256
max_iters = 10
eval_interval = 2
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

In [3]:
DATASETS = {
    "shakespeare": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "sherlock": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "pride_prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
    "alice": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "frankenstein": "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "dracula": "https://www.gutenberg.org/cache/epub/345/pg345.txt",
    "grimm": "https://www.gutenberg.org/cache/epub/1468/pg1468.txt",
    "tom_sawyer": "https://www.gutenberg.org/cache/epub/74/pg74.txt",
    "ulysses": "https://www.gutenberg.org/files/4300/4300-0.txt",
    "metamorphosis": "https://www.gutenberg.org/cache/epub/5200/pg5200.txt"
}

In [4]:
def clean_gutenberg_text(text):
    start_mark = "*** START OF"
    end_mark = "*** END OF"
    start_idx = text.find(start_mark)
    if start_idx != -1:
        text = text[text.find("\n", start_idx + len(start_mark)):]
    end_idx = text.find(end_mark)
    if end_idx != -1:
        text = text[:end_idx]
    return text.strip()

def prepare_data(output_file="chat_corpus.txt"):
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            return f.read()

    combined_text = ""
    for name, url in DATASETS.items():
        try:
            raw_text = requests.get(url).text
            clean_text = clean_gutenberg_text(raw_text) if "gutenberg" in url else raw_text
            combined_text += clean_text + "\n\n"
        except:
            print(f"err: {name}")

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(combined_text)
    return combined_text

In [5]:
text = prepare_data()
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [6]:
def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

In [7]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class SimpleGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [8]:
model = SimpleGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [9]:
for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.3f}, val loss {losses['val']:.3f}")

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("\n" + "="*30)
user_prompt = input("prompt")

context = torch.tensor(encode(user_prompt), dtype=torch.long, device=device).unsqueeze(0)

print(f"\n'{user_prompt}' model response:\n")
print("-" * 30)

generated_output = model.generate(context, max_new_tokens=300)
print(decode(generated_output[0].tolist()))
print("-" * 30)

step 0: train loss 5.066, val loss 5.066
step 2: train loss 3.609, val loss 3.629
step 4: train loss 3.288, val loss 3.301
step 6: train loss 3.167, val loss 3.179
step 8: train loss 3.074, val loss 3.088

prompthello

'hello' model response:

------------------------------
hellonO
d g
hy tÈjqeād
māiolr
iB Ttad t t br ra g ,Gy st t +o tnk ih os tstia ssrhecza, 
loiog✠o:fd so
------------------------------
